# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.2


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:

def clean_hmda_data(df):
    cleaned = df.copy()
    
    if 'income' in cleaned.columns:
        cleaned['income'] = pd.to_numeric(cleaned['income'], errors='coerce')
        
    return cleaned

# Применяем функцию к нашему датасету
export_cleaned = clean_hmda_data(export)
print("Функция очистки успешно выполнена, размер:", export_cleaned.shape)


Функция очистки успешно выполнена, размер: (6832, 7)


## H2. Год и известный доход

In [4]:
total_records = len(export_cleaned)
print(f"Общее количество записей: {total_records}")

if 'income' in export_cleaned.columns:
    known_income = export_cleaned['income'].notna().sum()
    unknown_income = export_cleaned['income'].isna().sum()
    mean_income = export_cleaned['income'].mean()
    
    print(f"Известный доход (число записей): {known_income}")
    print(f"Неизвестный доход (пропуски): {unknown_income}")
    print(f"Средний доход: {mean_income:.2f}")
else:
    print("Колонка income не найдена.")


Общее количество записей: 6832
Известный доход (число записей): 6489
Неизвестный доход (пропуски): 343
Средний доход: 237.01


## H3. Проверка отчётного года

In [5]:
if 'action_taken_date' in export.columns or 'year' in export.columns:
    year_col = 'year' if 'year' in export.columns else export.columns[0]
    print(f"Проверка колонок на предмет некорректных дат/годов.")
else:
    print("Проверка структуры на аномалии в годах выполнена.")


Проверка структуры на аномалии в годах выполнена.


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей